# MyoMap AI — Kaggle Train (Internet ON)
RSNA Knee 12-label multimodal. P100 16GB tuned. Enable Internet in Notebook Settings → Internet ON.
Dataset: Add `rsna-knee-abnormality-detection` as input. This notebook lives in `/kaggle/working/myomap-ai` if uploaded as dataset.

In [ ]:
# 0 — Env check + Kaggle Internet
import sys, os
from pathlib import Path
!nvidia-smi 2>&1 | head -n 20
print(f"python {sys.version}")
print(f"kaggle input exists: {Path('/kaggle/input/rsna-knee-abnormality-detection').exists()}")
print(f"working: {Path('/kaggle/working').exists()}")
# Internet check
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=5)
    print("Internet ON ✓")
except Exception as e:
    print(f"Internet OFF — enable in Settings → Internet ON. Err: {e}")


In [ ]:
# 1 — Install deps (Kaggle internet → pip, cached after first run)
# Run once per session with Internet ON. On rerun with Internet OFF, skip if already installed.
!pip install -q --upgrade pip
!pip install -q timm==1.0.9 transformers==4.44.2 albumentations==1.4.0 pydicom==2.4.4 pylibjpeg==1.4.0 pylibjpeg-libjpeg==1.3.2 accelerate==0.33.0 opencv-python-headless==4.10.0.84 2>&1 | tail -n 5
print("deps installed")


In [ ]:
# 2 — Locate myomap-ai code
# Option A: code uploaded as Kaggle Dataset `myomap-ai`
# Option B: clone from git / unzip. This handles both.
import shutil
for cand in [Path("/kaggle/input/myomap-ai"), Path("myomap-ai"), Path("/kaggle/working/myomap-ai")]:
    print(f"{cand} -> {cand.exists()}")
    if cand.exists():
        print(list(cand.rglob("*.py"))[:5])
        break
# If not found, expect you uploaded myomap-ai folder to /kaggle/input or /kaggle/working
CODE_ROOT = Path("myomap-ai") if Path("myomap-ai/src/train.py").exists() else Path("/kaggle/input/myomap-ai")
if not CODE_ROOT.exists():
    CODE_ROOT = Path("/kaggle/working")
print(f"CODE_ROOT={CODE_ROOT}")


In [ ]:
# 3 — EDA quick
import pandas as pd
from pathlib import Path
ROOT = Path("/kaggle/input/rsna-knee-abnormality-detection")
csvs = list(ROOT.rglob("*.csv"))
print(csvs[:10])
if csvs:
    df = pd.read_csv(csvs[0])
    print(df.head())
    print(df.shape)
    label_cols = [c for c in df.columns if c not in ["StudyInstanceUID","report_text","report","fold","StudyId"]]
    print("candidate labels", label_cols[:15])
    try:
        print(df[label_cols].mean().sort_values())
    except: pass


In [ ]:
# 4 — Train one fold (uses internet to download convnext + xlm-roberta weights first time)
# Ensure internet ON for first run — weights cached to /root/.cache/huggingface
import sys
sys.path.insert(0, str(CODE_ROOT / "src"))
if not (CODE_ROOT / "src" / "train.py").exists():
    # try working copy
    CODE_ROOT = Path("myomap-ai")
    sys.path.insert(0, str(CODE_ROOT / "src"))
print(f"train.py exists: {(CODE_ROOT / 'src' / 'train.py').exists()}")
!ls -R myomap-ai 2>&1 | head -n 60


In [ ]:
# actual train — adjust config path
CONFIG = str(CODE_ROOT / "configs" / "config.yaml")
print(CONFIG)
!python myomap-ai/src/train.py --config myomap-ai/configs/config.yaml --fold 0 2>&1 | tee /kaggle/working/train.log


In [ ]:
# 5 — Evaluate + Submission
!python myomap-ai/src/evaluate.py --ckpt myomap-ai/models/best_fold0.pth --config myomap-ai/configs/config.yaml --fold 0 2>&1 | tail -n 40
!python myomap-ai/scripts/submission.py --ckpt myomap-ai/models/best_fold0.pth --config myomap-ai/configs/config.yaml --out /kaggle/working/submission.csv 2>&1 | tail -n 30
!head /kaggle/working/submission.csv
!wc -l /kaggle/working/submission.csv


## Notes
- First run with Internet ON downloads ~300MB (convnextv2_tiny.fcmae ~100MB + xlm-roberta-base ~1GB). Cache persists in session.
- For scored submission, disable Internet after weights cached if you need to run with Internet OFF for final submission (optional). This notebook runs with Internet ON per your request.
- To train 5 folds: loop `--fold 0..4`, then ensemble by averaging probs in submission.
- Models saved to `/kaggle/working/myomap-ai/models/` — save as Kaggle Dataset for reuse.